## Previous_application data cleaning

In [20]:
import pandas as pd
previous = pd.read_csv("C:/Users/carla/Documents/IRONHACK/Week4/home-credit-default-risk/previous_application.csv")

In [21]:
previous.shape

(1670214, 37)

In [22]:
previous_columns = [
    'SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'CODE_REJECT_REASON',
    'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'NAME_YIELD_GROUP',
    'CHANNEL_TYPE', 'NAME_SELLER_INDUSTRY'
]
previous = previous[previous_columns]
previous.shape

(1670214, 9)

**1. Missing values**

In [23]:
previous.isnull().sum()

SK_ID_PREV              0
SK_ID_CURR              0
NAME_CONTRACT_STATUS    0
CODE_REJECT_REASON      0
NAME_PORTFOLIO          0
NAME_PRODUCT_TYPE       0
NAME_YIELD_GROUP        0
CHANNEL_TYPE            0
NAME_SELLER_INDUSTRY    0
dtype: int64

**2. Duplicates**

In [24]:
previous.duplicated().sum()

np.int64(0)

In [25]:
# check for duplicated ID keys
previous['SK_ID_PREV'].duplicated().sum()

np.int64(0)

In [26]:
previous.info()

<class 'pandas.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype
---  ------                --------------    -----
 0   SK_ID_PREV            1670214 non-null  int64
 1   SK_ID_CURR            1670214 non-null  int64
 2   NAME_CONTRACT_STATUS  1670214 non-null  str  
 3   CODE_REJECT_REASON    1670214 non-null  str  
 4   NAME_PORTFOLIO        1670214 non-null  str  
 5   NAME_PRODUCT_TYPE     1670214 non-null  str  
 6   NAME_YIELD_GROUP      1670214 non-null  str  
 7   CHANNEL_TYPE          1670214 non-null  str  
 8   NAME_SELLER_INDUSTRY  1670214 non-null  str  
dtypes: int64(2), str(7)
memory usage: 114.7 MB


**3. Categorical variables**

In [27]:
categorical_columns = ['NAME_CONTRACT_STATUS', 
                       'CODE_REJECT_REASON', 
                       'NAME_PORTFOLIO',
                       'NAME_PRODUCT_TYPE',
                       'NAME_YIELD_GROUP',
                       'CHANNEL_TYPE',
                       'NAME_SELLER_INDUSTRY']

for column in categorical_columns:
    print()
    print(previous[column].value_counts())


NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

CODE_REJECT_REASON
XAP       1353093
HC         175231
LIMIT       55680
SCO         37467
CLIENT      26436
SCOFR       12811
XNA          5244
VERIF        3535
SYSTEM        717
Name: count, dtype: int64

NAME_PORTFOLIO
POS      691011
Cash     461563
XNA      372230
Cards    144985
Cars        425
Name: count, dtype: int64

NAME_PRODUCT_TYPE
XNA        1063666
x-sell      456287
walk-in     150261
Name: count, dtype: int64

NAME_YIELD_GROUP
XNA           517215
middle        385532
high          353331
low_normal    322095
low_action     92041
Name: count, dtype: int64

CHANNEL_TYPE
Credit and cash offices       719968
Country-wide                  494690
Stone                         212083
Regional / Local              108528
Contact center                 71297
AP+ (Cash loan)                57046
Channel of corporate sales      6150
Ca

In [28]:
xna_columns = [
    'CODE_REJECT_REASON',
    'NAME_PORTFOLIO',
    'NAME_PRODUCT_TYPE',
    'NAME_YIELD_GROUP',
    'NAME_SELLER_INDUSTRY'
]
xna_count = (previous[xna_columns] == "XNA").sum(axis=1)
(xna_count == 5).sum()

np.int64(278)

In [29]:
xna_all = previous[xna_count == 5]
xna_all.head()
#xna_all['NAME_CONTRACT_STATUS'].value_counts()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_STATUS,CODE_REJECT_REASON,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,NAME_YIELD_GROUP,CHANNEL_TYPE,NAME_SELLER_INDUSTRY
7297,1068523,383080,Refused,XNA,XNA,XNA,XNA,Credit and cash offices,XNA
8793,2376924,378705,Refused,XNA,XNA,XNA,XNA,Credit and cash offices,XNA
17638,1117592,293632,Refused,XNA,XNA,XNA,XNA,Credit and cash offices,XNA
19677,1089536,138942,Refused,XNA,XNA,XNA,XNA,Credit and cash offices,XNA
20228,1145005,363608,Refused,XNA,XNA,XNA,XNA,Credit and cash offices,XNA


**XNA values**

We identified 278 rows where all five variables than can contain 'XNA' have this value.

All 278 records have 'NAME_CONTRACT_STATUS = 'Refused''. Therefore, these 'XNA' values seems to be associated with refused applications rather than being random missing data.

We decided to keep these records because they contain relevant information about previous rejected applications???

**3. ID consistency**

In [30]:
previous.groupby('SK_ID_PREV')['SK_ID_CURR'].nunique().value_counts()

SK_ID_CURR
1    1670214
Name: count, dtype: int64

In [31]:
previous_agg = (previous
    .groupby('SK_ID_CURR')
    .agg(
        total_previous_applications = ('SK_ID_PREV', 'count'),
        approved_applications=('NAME_CONTRACT_STATUS', lambda x: (x=='Approved').sum()),
        refused_applications=('NAME_CONTRACT_STATUS', lambda x: (x=='Refused').sum()),
        canceled_applications=('NAME_CONTRACT_STATUS', lambda x: (x=='Canceled').sum()),
        unused_offers=('NAME_CONTRACT_STATUS', lambda x: (x=='Unused offer').sum())
    )
    .reset_index()
)

In [32]:
previous_agg

,SK_ID_CURR,total_previous_applications,approved_applications,refused_applications,canceled_applications,unused_offers
0,100001,1,1,0,0,0
1,100002,1,1,0,0,0
2,100003,3,3,0,0,0
3,100004,1,1,0,0,0
4,100005,2,1,0,1,0
...,...,...,...,...,...,...
338852,456251,1,1,0,0,0
338853,456252,1,1,0,0,0
338854,456253,2,2,0,0,0
338855,456254,2,2,0,0,0


In [ ]:
# no funciona 
# previous.to_csv(config['output_data']['previous_application'], index=False)